# 72 ? Experimento TF-IDF + DBSCAN

Este notebook testa agrupamento de descri??es normalizadas de itens de supermercado usando TF-IDF e DBSCAN. A an?lise usa dist?ncia cosseno, n?o usa NCM nem r?tulos humanos, e trata ru?do como resultado expl?cito do algoritmo. A amostra configur?vel mant?m o custo do c?lculo de vizinhan?a controlado. Execute de cima para baixo em uma sess?o limpa.


In [27]:
from datetime import datetime, timezone
from pathlib import Path
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from matplotlib.colors import ListedColormap

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook72")
pd.set_option("display.max_colwidth", 140)


## 1. Parametros

`eps` e o raio maximo de vizinhanca na distancia cosseno; `min_samples` e o numero minimo de pontos para formar uma regiao densa. A escolha e sensivel a escala de `eps`, entao a grade inicial e exploratoria. Silhouette sera calculada somente nos itens nao classificados como ruido e nao representa qualidade semantica nem acuracia.

A analise cruza duas sementes de amostragem com dois analyzers. Para isolar os efeitos, cada analyzer usa a mesma amostra da sua semente. O analyzer `word` usa unigramas e bigramas, enquanto `char_wb` usa n-gramas de caracteres. A grade de `eps` e exploratoria para ambos; as distancias podem ter escalas diferentes.


`CONSIDER_FREQUENCY_IN_CLUSTERING` define se a frequencia pondera o DBSCAN e fica False por padrao. Assim, cada descricao unica conta uma vez; depois, a frequencia ordena os clusters pelo volume e as descricoes unitarias pela sua frequencia.


In [28]:
DATASET_NAME = "2025_NCMs_filtrados-sem-duplicatas"  # Nome da pasta em data/processed; altere apenas este valor para trocar a base.


PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nao encontrei a raiz do repositorio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/periodo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve comecar com '{DATASET_NAME}-' para derivar o periodo: "
        f"{DATASET_PATH.name}"
    )



PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MAX_DESCRIPTIONS = 5_000  # Amostra reprodutivel; DBSCAN tem custo quadratico em memoria.
CONSIDER_FREQUENCY_IN_CLUSTERING = False  # False: uma observacao por descricao; True: pondera por frequencia.
SAMPLE_SEEDS = [42]
ANALYZER_CONFIGS = {
    "char_wb_3_5": ("char_wb", (3, 5)),
    "word_1_2": ("word", (1, 2)),
    "word_1_3": ("word", (1, 3)),
}
MIN_DF = 20
MAX_FEATURES = 100_000
DBSCAN_N_JOBS = 1  # Reduz o pico de memoria no calculo de vizinhancas.
EPS_VALUES = [0.03, 1, 5, 20, 100]
MIN_SAMPLES_VALUES = [3, 5, 10, 20, 30, 100]
OUTPUT_ROOT = Path("reports/tfidf_dbscan")

TSNE_MAX_POINTS = 5_000  # Amostra fixa para limitar o custo da proje??o.
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
TSNE_SVD_COMPONENTS = 50  # Redu??o pr?via para matrizes TF-IDF esparsas.

## 2. Fun??es da an?lise

A prepara??o remove descri??es vazias, mant?m uma linha por descri??o e valida que a frequ?ncia total n?o foi alterada. Cada configura??o ? executada em uma c?pia l?gica da mesma matriz TF-IDF.


In [29]:
def prepare_items(frame):
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = (
        prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY]
        .sum()
    )
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)

def summarize_clusters(items, labels):
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary

def score_configuration(matrix, labels):
    labels = np.asarray(labels)
    noise_mask = labels == -1
    clustered_labels = labels[~noise_mask]
    cluster_count = len(np.unique(clustered_labels))
    score = np.nan
    if cluster_count >= 2 and cluster_count < len(clustered_labels):
        score = float(silhouette_score(matrix[~noise_mask], clustered_labels, metric="cosine"))
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_cosine_without_noise": score,
    }


def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state):
    """Calcula uma proje??o t-SNE reprodut?vel sobre uma amostra das linhas."""
    point_count = features.shape[0]
    if point_count < 3:
        raise ValueError("O t-SNE requer pelo menos tr?s descri??es.")

    sample_count = min(point_count, max_points)
    if sample_count < 3:
        raise ValueError("TSNE_MAX_POINTS deve ser pelo menos tres.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled_features = features[indices]

    if sampled_features.shape[1] > 1:
        component_count = min(TSNE_SVD_COMPONENTS, sampled_features.shape[1] - 1, sample_count - 1)
        reduced_features = TruncatedSVD(
            n_components=component_count, random_state=random_state,
        ).fit_transform(sampled_features)
    else:
        reduced_features = sampled_features.toarray()

    effective_perplexity = min(float(perplexity), (sample_count - 1) / 3)
    coordinates = TSNE(
        n_components=2,
        perplexity=effective_perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=max_iter,
        random_state=random_state,
        method="barnes_hut",
    ).fit_transform(reduced_features)
    return indices, coordinates, reduced_features.shape[1]


def save_tsne_plot(coordinates, labels, run_id, output_path):
    """Salva um PNG t-SNE colorido pelos clusters, com o ru?do em cinza."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_ids = np.unique(labels[~noise_mask])
    if cluster_ids.size:
        palette = plt.get_cmap("tab20").colors
        cluster_cmap = ListedColormap([
            palette[index % len(palette)] for index in range(len(cluster_ids))
        ])
        cluster_codes = np.searchsorted(cluster_ids, labels[~noise_mask])
    else:
        cluster_cmap = None
        cluster_codes = np.empty(0, dtype=int)

    figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
    if noise_mask.any():
        axis.scatter(
            coordinates[noise_mask, 0], coordinates[noise_mask, 1],
            c="#b8b8b8", s=8, alpha=0.3, linewidths=0,
        )
    if cluster_ids.size:
        axis.scatter(
            coordinates[~noise_mask, 0], coordinates[~noise_mask, 1],
            c=cluster_codes, cmap=cluster_cmap, vmin=-0.5,
            vmax=len(cluster_ids) - 0.5, s=9, alpha=0.75, linewidths=0,
        )
    axis.set_title(
        f"t-SNE: {run_id}\n{len(cluster_ids)} clusters; "
        f"{int(noise_mask.sum()):,} descri??es como ru?do"
    )
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.text(
        0.01, 0.01,
        "Ru?do em cinza; as cores se repetem a cada 20 clusters.",
        transform=axis.transAxes, fontsize=8, color="#555555",
    )
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


## 3. Carregar e preparar dados

Origem: Parquet produzido pelos notebooks de preparacao de dados do projeto, com descricoes de supermercado, NCMs e duplicatas agregadas por descricao. A amostragem e feita uma vez por semente e reutilizada entre os analyzers daquela semente.


In [30]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))


2026-10-15 13:35:04,899 INFO Descricoes disponiveis apos preparacao: 149149


## 4. Vetorizar, ajustar DBSCAN e comparar parametros

Cada combinacao de semente e analyzer gera uma matriz TF-IDF propria. `metric="cosine"` permite trabalhar diretamente com a matriz esparsa. O ruido (`cluster = -1`) e excluido da silhouette e contabilizado separadamente; configuracoes com menos de dois clusters validos ficam sem silhouette. A matriz completa nao e convertida para densa.

Para visualizacao, uma amostra fixa de ate `TSNE_MAX_POINTS` descricoes e reduzida com TruncatedSVD e projetada uma vez por representacao. As coordenadas sao reutilizadas para salvar um PNG por combinacao de parametros.


In [31]:
records = []
labels_by_run = {}
items_by_run = {}
tsne_by_representation = {}
items_by_seed = {}

for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    items_by_seed[sample_seed] = sampled_items

    for representation, (analyzer, ngram_range) in ANALYZER_CONFIGS.items():
        logger.info(
            "Vetorizando analyzer=%s, seed=%d, descricoes=%d",
            analyzer, sample_seed, len(sampled_items),
        )
        vectorizer = TfidfVectorizer(
            analyzer=analyzer, ngram_range=ngram_range, min_df=MIN_DF,
            max_features=MAX_FEATURES, sublinear_tf=True, norm="l2",
        )
        matrix = vectorizer.fit_transform(sampled_items[DESCRIPTION_NORMALIZED].astype(str))
        zero_vector_count = int((matrix.getnnz(axis=1) == 0).sum())
        logger.info("Calculando t-SNE para %s, seed=%d", representation, sample_seed)
        tsne_indices, tsne_coordinates, tsne_dimensions = compute_tsne_projection(
            matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
            TSNE_RANDOM_STATE + sample_seed,
        )
        tsne_by_representation[(sample_seed, representation)] = {
            "indices": tsne_indices, "coordinates": tsne_coordinates,
            "reduced_dimensions": tsne_dimensions,
        }
        logger.info(
            "Analyzer=%s seed=%d: %d termos, %d vetores zerados",
            analyzer, sample_seed, matrix.shape[1], zero_vector_count,
        )

        for eps in EPS_VALUES:
            for min_samples in MIN_SAMPLES_VALUES:
                run_id = f"seed{sample_seed}_{representation}_eps{eps:g}_min_samples{min_samples}"
                logger.info("Ajustando DBSCAN %s", run_id)
                model = DBSCAN(
                    eps=eps, min_samples=min_samples, metric="cosine",
                    algorithm="brute", n_jobs=DBSCAN_N_JOBS,
                )
                sample_weight = (
                    sampled_items[FREQUENCY].to_numpy(dtype=float)
                    if CONSIDER_FREQUENCY_IN_CLUSTERING else None
                )
                model.fit(matrix, sample_weight=sample_weight)
                labels = model.labels_
                metrics = score_configuration(matrix, labels)
                records.append({
                    "run_id": run_id, "sample_seed": sample_seed,
                    "representation": representation, "analyzer": analyzer,
                    "ngram_range": str(ngram_range),
                    "eps": eps, "min_samples": min_samples,
                    "frequency_weighted": CONSIDER_FREQUENCY_IN_CLUSTERING, **metrics,
                })
                labels_by_run[run_id] = labels
                items_by_run[run_id] = sampled_items

results = pd.DataFrame(records).sort_values(
    ["silhouette_cosine_without_noise", "noise_fraction"],
    ascending=[False, True], na_position="last",
).reset_index(drop=True)
display(results)


2026-10-15 13:35:04,964 INFO Vetorizando analyzer=char_wb, seed=42, descricoes=5000
2026-10-15 13:35:05,563 INFO Calculando t-SNE para char_wb_3_5, seed=42
2026-10-15 13:35:43,273 INFO Analyzer=char_wb seed=42: 3592 termos, 0 vetores zerados
2026-10-15 13:35:43,275 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps0.03_min_samples3
2026-10-15 13:35:43,884 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps0.03_min_samples5
2026-10-15 13:35:44,454 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps0.03_min_samples10
2026-10-15 13:35:44,920 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps0.03_min_samples20
2026-10-15 13:35:45,375 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps0.03_min_samples30
2026-10-15 13:35:45,845 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps0.03_min_samples100
2026-10-15 13:35:46,338 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps1_min_samples3
2026-10-15 13:35:47,230 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps1_min_samples5
2026-10-15 13:35:48,159 INFO Ajustando DBSCAN seed42_char_wb_3_5_eps1_m

,run_id,sample_seed,representation,analyzer,ngram_range,eps,min_samples,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_cosine_without_noise
0,seed42_word_1_2_eps0.03_min_samples20,42,word_1_2,word,"(1, 2)",0.03,20,False,4,4831,0.9662,1.000000
1,seed42_word_1_3_eps0.03_min_samples20,42,word_1_3,word,"(1, 3)",0.03,20,False,4,4831,0.9662,1.000000
2,seed42_word_1_2_eps0.03_min_samples30,42,word_1_2,word,"(1, 2)",0.03,30,False,2,4880,0.9760,1.000000
3,seed42_word_1_3_eps0.03_min_samples30,42,word_1_3,word,"(1, 3)",0.03,30,False,2,4880,0.9760,1.000000
4,seed42_word_1_2_eps0.03_min_samples10,42,word_1_2,word,"(1, 2)",0.03,10,False,19,4653,0.9306,0.999693
...,...,...,...,...,...,...,...,...,...,...,...,...
85,seed42_char_wb_3_5_eps0.03_min_samples20,42,char_wb_3_5,char_wb,"(3, 5)",0.03,20,False,0,5000,1.0000,NaN
86,seed42_char_wb_3_5_eps0.03_min_samples30,42,char_wb_3_5,char_wb,"(3, 5)",0.03,30,False,0,5000,1.0000,NaN
87,seed42_char_wb_3_5_eps0.03_min_samples100,42,char_wb_3_5,char_wb,"(3, 5)",0.03,100,False,0,5000,1.0000,NaN
88,seed42_word_1_2_eps0.03_min_samples100,42,word_1_2,word,"(1, 2)",0.03,100,False,0,5000,1.0000,NaN


In [34]:
results.head(20)

,run_id,sample_seed,representation,analyzer,ngram_range,eps,min_samples,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_cosine_without_noise
0,seed42_word_1_2_eps0.03_min_samples20,42,word_1_2,word,"(1, 2)",0.03,20,False,4,4831,0.9662,1.000000
1,seed42_word_1_3_eps0.03_min_samples20,42,word_1_3,word,"(1, 3)",0.03,20,False,4,4831,0.9662,1.000000
2,seed42_word_1_2_eps0.03_min_samples30,42,word_1_2,word,"(1, 2)",0.03,30,False,2,4880,0.9760,1.000000
3,seed42_word_1_3_eps0.03_min_samples30,42,word_1_3,word,"(1, 3)",0.03,30,False,2,4880,0.9760,1.000000
4,seed42_word_1_2_eps0.03_min_samples10,42,word_1_2,word,"(1, 2)",0.03,10,False,19,4653,0.9306,0.999693
5,seed42_word_1_3_eps0.03_min_samples10,42,word_1_3,word,"(1, 3)",0.03,10,False,19,4653,0.9306,0.999693
6,seed42_word_1_2_eps0.03_min_samples5,42,word_1_2,word,"(1, 2)",0.03,5,False,113,4069,0.8138,0.999462
7,seed42_word_1_3_eps0.03_min_samples5,42,word_1_3,word,"(1, 3)",0.03,5,False,113,4069,0.8138,0.999462
8,seed42_word_1_3_eps0.03_min_samples3,42,word_1_3,word,"(1, 3)",0.03,3,False,264,3563,0.7126,0.997926
9,seed42_word_1_2_eps0.03_min_samples3,42,word_1_2,word,"(1, 2)",0.03,3,False,264,3563,0.7126,0.997922


## 5. Revisar um resultado e salvar relat?rios

Escolha `SELECTED_RUN_ID` ap?s comparar quantidade de clusters, ru?do e silhouette. A tabela de resumo inclui `-1` como grupo de ru?do. Os arquivos s?o gravados em uma pasta com identificador UTC ?nico para preservar execu??es anteriores.

Os graficos ficam em `tsne_plots/`, um PNG por combinacao, e as coordenadas da amostra em `tsne_coordinates/`, dentro da pasta da execucao.


In [32]:
SELECTED_RUN_ID = results.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
cluster_summary = cluster_summary.sort_values("frequencia_total", ascending=False).reset_index(drop=True)
cluster_sizes = assignments.loc[assignments["cluster"].ne(-1)].groupby("cluster")[DESCRIPTION_NORMALIZED].size()
singleton_clusters = cluster_sizes[cluster_sizes.eq(1)].index
frequent_singletons = assignments[assignments["cluster"].isin(singleton_clusters)].sort_values(
    FREQUENCY, ascending=False
)
frequent_noise = assignments[assignments["cluster"].eq(-1)].sort_values(
    FREQUENCY, ascending=False
)
display(cluster_summary.head(20))
display(frequent_singletons.head(20))
display(frequent_noise.head(20))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
frequent_singletons.to_csv(
    output_dir / "frequent_singleton_descriptions.csv", index=False, encoding="utf-8-sig"
)
frequent_noise.to_csv(
    output_dir / "frequent_noise_descriptions.csv", index=False, encoding="utf-8-sig"
)

# A proje??o ? compartilhada pelos valores de eps/min_samples da mesma representa??o.
tsne_coordinates_dir = output_dir / "tsne_coordinates"
tsne_plots_dir = output_dir / "tsne_plots"
tsne_coordinates_dir.mkdir()
tsne_plots_dir.mkdir()
tsne_artifacts = {}
for (sample_seed, representation), projection in tsne_by_representation.items():
    indices = projection["indices"]
    coordinates = projection["coordinates"]
    coordinate_frame = items_by_seed[sample_seed].iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    coordinate_frame["tsne_1"] = coordinates[:, 0]
    coordinate_frame["tsne_2"] = coordinates[:, 1]
    coordinate_name = f"seed{sample_seed}_{representation}_coordinates.csv"
    coordinate_frame.to_csv(
        tsne_coordinates_dir / coordinate_name, index=False, encoding="utf-8-sig",
    )
    tsne_artifacts[f"seed{sample_seed}_{representation}"] = {
        "coordinates_file": f"tsne_coordinates/{coordinate_name}",
        "sample_size": len(indices),
        "reduced_dimensions": projection["reduced_dimensions"],
    }

for run_id, labels in labels_by_run.items():
    result_row = results.loc[results["run_id"].eq(run_id)].iloc[0]
    projection = tsne_by_representation[(int(result_row["sample_seed"]), result_row["representation"])]
    plot_labels = np.asarray(labels)[projection["indices"]]
    plot_path = tsne_plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], plot_labels, run_id, plot_path)
    tsne_artifacts[f"plot:{run_id}"] = str(plot_path.relative_to(output_dir))

logger.info("Imagens t-SNE salvas: %d", len(labels_by_run))
run_metadata = {
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "frequency_weighted": CONSIDER_FREQUENCY_IN_CLUSTERING,
    "dataset_path": str(DATASET_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "analyzer_configs": {
        representation: {"analyzer": analyzer, "ngram_range": list(ngram_range)}
        for representation, (analyzer, ngram_range) in ANALYZER_CONFIGS.items()
    },
    "min_df": MIN_DF, "max_features": MAX_FEATURES,
    "selected_run_id": SELECTED_RUN_ID,
    "eps_values": EPS_VALUES, "min_samples_values": MIN_SAMPLES_VALUES,
    "tsne": {
        "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
        "svd_components": TSNE_SVD_COMPONENTS, "artifacts": tsne_artifacts,
    },
}
(output_dir / "run.json").write_text(
    __import__("json").dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Relatorios e imagens t-SNE salvos em %s", output_dir)


,cluster,descricoes_unicas,frequencia_total
0,-1,4831,25376
1,1,39,1021
2,0,81,557
3,2,27,201
4,3,22,151


,descricao_normalizada,frequencia,cluster,ruido


,descricao_normalizada,frequencia,cluster,ruido
3522,paninho de limpeza 28x38,614,-1,True
1435,cola liquida branca go office 90g,514,-1,True
3256,molho ingles,463,-1,True
263,agua mineral potiguar 20 lts liquido,391,-1,True
1773,detergente,346,-1,True
3014,macarrao espaguete,338,-1,True
1100,caixa plastica,290,-1,True
2853,linguica,256,-1,True
4567,sardinha conserva oleo 125g,255,-1,True
3054,mamao,242,-1,True


2026-10-15 13:38:35,985 INFO Imagens t-SNE salvas: 90
2026-10-15 13:38:36,000 INFO Relatorios e imagens t-SNE salvos em reports\tfidf_dbscan\202501-202512_20261015T163751Z


### Interpreta??o

- `noise_fraction` alta significa que muitas descri??es n?o entraram em regi?es densas para os par?metros escolhidos.
- Poucos clusters ou um cluster dominante podem indicar que `eps` est? grande; quase tudo como ru?do pode indicar `eps` pequeno ou `min_samples` alto.
- Silhouette mais alta deve ser lida junto da fra??o de ru?do: uma pontua??o calculada em poucos itens pode ser enganosa.
- DBSCAN n?o atribui automaticamente novos produtos a clusters ap?s o ajuste e n?o define categorias sem revis?o humana.
